#place_lat & plat_lon

In [1]:
pip install pandas


Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install geopy

Note: you may need to restart the kernel to use updated packages.


In [4]:
#take me back back home


import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter    
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
import re


#1 load data
df = pd.read_csv('Phivolcs_Master_Aggregated_Fixed_FinalFinished.csv')
df

,event_id,date,time,latitude,longitude,depth_km,mag,coordinates,Location,loc_latitude,loc_longitude,city,province,region,distance_km,ns,angle_deg,ew
0,PHV-0001,2/15/2026,1:22:00,5.31,127.44,18,5.3,209 km S 71Â° E,Jose Abad Santos (Davao Occidental),5.912465,125.644593,Jose Abad Santos,Davao Occidental,Davao Region (Region XI),209.0,S,71.0,E
1,PHV-0002,2/13/2026,22:56:00,3.95,128.62,53,5.6,387 km S 64Â° E,Sarangani Island (Municipality Of Sarangani) (...,5.442653,125.475756,Sarangani,Davao Occidental,Davao Region (Region XI),387.0,S,64.0,E
2,PHV-0003,2/13/2026,23:55:00,4.22,127.84,62,5.1,298 km S 62Â° E,Sarangani Island (Municipality Of Sarangani) (...,5.442653,125.475756,Sarangani,Davao Occidental,Davao Region (Region XI),298.0,S,62.0,E
3,PHV-0004,2/12/2026,6:27:00,20.60,119.99,32,5.3,194 km S 83Â° W,Itbayat (Batanes),20.787814,121.841474,Itbayat,Batanes,Cagayan Valley (Region II),194.0,S,83.0,W
4,PHV-0005,1/28/2026,16:34:00,6.49,123.67,16,5.2,043 km S 80Â° W,Kalamansig (Sultan Kudarat),6.554584,124.050599,Kalamansig,Sultan Kudarat,SOCCSKSARGEN (Region XII),43.0,S,80.0,W
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
766,PHV-0767,11/5/2010,NaN,12.86,122.85,21,5.0,28 km SSW,"San Rafael, Masbate",13.097351,122.960070,"San Rafael, Masbate",Masbate,Bicol Region (Region V),NaN,NaN,NaN,NaN
767,PHV-0768,7/23/2010,NaN,7.32,123.77,602,7.4,NaN,"Moro Gulf, Mindanao",7.333402,123.783090,"Moro Gulf, Mindanao",NaN,"Regions IX, XII, & BARMM",NaN,NaN,NaN,NaN
768,PHV-0769,7/23/2010,NaN,6.50,123.48,578,7.6,NaN,"Moro Gulf, Mindanao",7.333402,123.783090,"Moro Gulf, Mindanao",NaN,"Regions IX, XII, & BARMM",NaN,NaN,NaN,NaN
769,PHV-0770,7/23/2010,NaN,6.72,123.41,607,7.3,NaN,"Moro Gulf, Mindanao",7.333402,123.783090,"Moro Gulf, Mindanao",NaN,"Regions IX, XII, & BARMM",NaN,NaN,NaN,NaN


In [8]:

#geoloctor
geolocator = Nominatim(user_agent="philippines_earthquake_mapper")

#rate limiter for api reqs
geocode_rate_limited = RateLimiter(geolocator.geocode, min_delay_seconds=1, max_retries=2)#1 sec

def clean_location(loc_string):
    if pd.isna(loc_string):
        return loc_string
    return re.sub(r'\s*\(', ', ', str(loc_string)).replace(')', '')

def safe_geocode(query):
    # Skip empty rows immediately
    if pd.isna(query) or query == "":
        return None
    
    try:
        # Attempt to geocode
        return geocode_rate_limited(query)
    except (GeocoderTimedOut, GeocoderServiceError) as e:
        print(f"API Error skipping '{query}': {e}")
        return None
    except Exception as e:
        print(f"Unexpected error skipping '{query}': {e}")
        return None



#cleaning function
df['Search_Query'] = df['Location'].apply(clean_location)

#apply
print("Geocoding...")
df['Geocode_Result'] = df['Search_Query'].apply(safe_geocode)

#extractor
df['loc_latitude'] = df['Geocode_Result'].apply(lambda loc: loc.latitude if loc else None)
df['loc_longitude'] = df['Geocode_Result'].apply(lambda loc: loc.longitude if loc else None)

#show
df = df.drop(columns=['Search_Query', 'Geocode_Result'])
df


Geocoding...


,date,time,latitude,longitude,depth_km,mag,coordinates,distance_km,ns,angle_deg,ew,Location,loc_latitude,loc_longitude
0,10/20/2025,19:35:00,7.00,127.36,20,5.7,094 km S 75° E,94,S,75,E,Manay (Davao Oriental),7.210933,126.539914
1,10/19/2025,9:34:00,6.95,127.36,20,5.2,095 km S 72° E,95,S,72,E,Manay (Davao Oriental),7.210933,126.539914
2,10/18/2025,20:31:00,8.84,126.45,34,5.5,019 km S 63° E,19,S,63,E,Cagwait (Surigao Del Sur),8.918274,126.298255
3,10/17/2025,7:03:00,9.69,126.22,28,6.0,012 km S 33° E,12,S,33,E,General Luna (Surigao Del Norte),9.782584,126.156386
4,10/14/2025,10:00:00,7.47,127.05,18,5.6,063 km N 63° E,63,N,63,E,Manay (Davao Oriental),7.210933,126.539914
5,10/13/2025,1:06:00,10.96,123.94,5,5.8,010 km S 24° W,10,S,24,W,City Of Bogo (Cebu),11.051555,124.003546
6,10/11/2025,22:32:00,8.94,126.55,24,6.0,028 km N 86° E,28,N,86,E,Cagwait (Surigao Del Sur),8.918274,126.298255
7,10/11/2025,18:27:00,7.23,126.91,18,5.5,041 km N 87° E,41,N,87,E,Manay (Davao Oriental),7.210933,126.539914
8,10/11/2025,17:32:00,15.24,120.23,112,5.1,022 km S 77° E,22,S,77,E,Botolan (Zambales),15.288997,120.024736
9,10/10/2025,19:12:00,7.07,126.90,37,6.8,043 km S 68° E,43,S,68,E,Manay (Davao Oriental),7.210933,126.539914


In [9]:
import pandas as pd
dfSave = 'Phivolcs_aggregated-new5only.csv'
df.to_csv(dfSave, index=False)

print(f"File successfully saved as: {dfSave}")
df

File successfully saved as: Phivolcs_aggregated-new5only.csv


,date,time,latitude,longitude,depth_km,mag,coordinates,distance_km,ns,angle_deg,ew,Location,loc_latitude,loc_longitude
0,10/20/2025,19:35:00,7.00,127.36,20,5.7,094 km S 75° E,94,S,75,E,Manay (Davao Oriental),7.210933,126.539914
1,10/19/2025,9:34:00,6.95,127.36,20,5.2,095 km S 72° E,95,S,72,E,Manay (Davao Oriental),7.210933,126.539914
2,10/18/2025,20:31:00,8.84,126.45,34,5.5,019 km S 63° E,19,S,63,E,Cagwait (Surigao Del Sur),8.918274,126.298255
3,10/17/2025,7:03:00,9.69,126.22,28,6.0,012 km S 33° E,12,S,33,E,General Luna (Surigao Del Norte),9.782584,126.156386
4,10/14/2025,10:00:00,7.47,127.05,18,5.6,063 km N 63° E,63,N,63,E,Manay (Davao Oriental),7.210933,126.539914
5,10/13/2025,1:06:00,10.96,123.94,5,5.8,010 km S 24° W,10,S,24,W,City Of Bogo (Cebu),11.051555,124.003546
6,10/11/2025,22:32:00,8.94,126.55,24,6.0,028 km N 86° E,28,N,86,E,Cagwait (Surigao Del Sur),8.918274,126.298255
7,10/11/2025,18:27:00,7.23,126.91,18,5.5,041 km N 87° E,41,N,87,E,Manay (Davao Oriental),7.210933,126.539914
8,10/11/2025,17:32:00,15.24,120.23,112,5.1,022 km S 77° E,22,S,77,E,Botolan (Zambales),15.288997,120.024736
9,10/10/2025,19:12:00,7.07,126.90,37,6.8,043 km S 68° E,43,S,68,E,Manay (Davao Oriental),7.210933,126.539914


# Location: City, Province Region

In [10]:
df1 = pd.read_csv('Phivolcs_aggregated-new5only.csv')
df1

,date,time,latitude,longitude,depth_km,mag,coordinates,distance_km,ns,angle_deg,ew,Location,loc_latitude,loc_longitude
0,10/20/2025,19:35:00,7.00,127.36,20,5.7,094 km S 75° E,94,S,75,E,Manay (Davao Oriental),7.210933,126.539914
1,10/19/2025,9:34:00,6.95,127.36,20,5.2,095 km S 72° E,95,S,72,E,Manay (Davao Oriental),7.210933,126.539914
2,10/18/2025,20:31:00,8.84,126.45,34,5.5,019 km S 63° E,19,S,63,E,Cagwait (Surigao Del Sur),8.918274,126.298255
3,10/17/2025,7:03:00,9.69,126.22,28,6.0,012 km S 33° E,12,S,33,E,General Luna (Surigao Del Norte),9.782584,126.156386
4,10/14/2025,10:00:00,7.47,127.05,18,5.6,063 km N 63° E,63,N,63,E,Manay (Davao Oriental),7.210933,126.539914
5,10/13/2025,1:06:00,10.96,123.94,5,5.8,010 km S 24° W,10,S,24,W,City Of Bogo (Cebu),11.051555,124.003546
6,10/11/2025,22:32:00,8.94,126.55,24,6.0,028 km N 86° E,28,N,86,E,Cagwait (Surigao Del Sur),8.918274,126.298255
7,10/11/2025,18:27:00,7.23,126.91,18,5.5,041 km N 87° E,41,N,87,E,Manay (Davao Oriental),7.210933,126.539914
8,10/11/2025,17:32:00,15.24,120.23,112,5.1,022 km S 77° E,22,S,77,E,Botolan (Zambales),15.288997,120.024736
9,10/10/2025,19:12:00,7.07,126.90,37,6.8,043 km S 68° E,43,S,68,E,Manay (Davao Oriental),7.210933,126.539914


In [11]:
def parse_location(loc_string):
    if pd.isna(loc_string):
        return pd.Series([None, None, None])
    
    loc_string = str(loc_string).strip()
    
    main_match = re.search(r'^[^\(]+', loc_string)
    main_name = main_match.group(0).strip() if main_match else loc_string
    
    parentheticals = re.findall(r'\((.*?)\)', loc_string)
    
    city = main_name
    province = None
    region = None # Blank, because it's not in the text
    
    if len(parentheticals) == 1:
        province = parentheticals[0].strip()
        
    elif len(parentheticals) >= 2:
        city = parentheticals[0].strip()     
        province = parentheticals[1].strip() 
        city = city.replace("Municipality Of ", "").replace("City Of ", "")

    return pd.Series([city, province, region])

# 4. Apply the function to df1
print("Parsing locations...")
df1[['city', 'province', 'region']] = df1['Location'].apply(parse_location)

# 5. Display the result from df1 to check the work
print(df1[['Location', 'city', 'province', 'region']].head(15))

Parsing locations...
                               Location             city           province  \
0                Manay (Davao Oriental)            Manay     Davao Oriental   
1                Manay (Davao Oriental)            Manay     Davao Oriental   
2             Cagwait (Surigao Del Sur)          Cagwait    Surigao Del Sur   
3      General Luna (Surigao Del Norte)     General Luna  Surigao Del Norte   
4                Manay (Davao Oriental)            Manay     Davao Oriental   
5                   City Of Bogo (Cebu)     City Of Bogo               Cebu   
6             Cagwait (Surigao Del Sur)          Cagwait    Surigao Del Sur   
7                Manay (Davao Oriental)            Manay     Davao Oriental   
8                    Botolan (Zambales)          Botolan           Zambales   
9                Manay (Davao Oriental)            Manay     Davao Oriental   
10               Manay (Davao Oriental)            Manay     Davao Oriental   
11               Manay (Davao O

In [12]:
dfSave = 'Phivolcs_aggregated_parsed-new5only.csv' 
df1.to_csv(dfSave, index=False)

print(f"File successfully saved as: {dfSave}")

File successfully saved as: Phivolcs_aggregated_parsed-new5only.csv


##Region

In [13]:
province_to_region = {
    # Region I (Ilocos Region)
    'Ilocos Norte': 'Ilocos Region (Region I)',
    'Ilocos Sur': 'Ilocos Region (Region I)',
    'La Union': 'Ilocos Region (Region I)',
    'Pangasinan': 'Ilocos Region (Region I)',

    # Region II (Cagayan Valley)
    'Batanes': 'Cagayan Valley (Region II)',
    'Cagayan': 'Cagayan Valley (Region II)',
    'Isabela': 'Cagayan Valley (Region II)',
    'Nueva Vizcaya': 'Cagayan Valley (Region II)',
    'Quirino': 'Cagayan Valley (Region II)',

    # Region III (Central Luzon)
    'Aurora': 'Central Luzon (Region III)',
    'Bataan': 'Central Luzon (Region III)',
    'Bulacan': 'Central Luzon (Region III)',
    'Nueva Ecija': 'Central Luzon (Region III)',
    'Pampanga': 'Central Luzon (Region III)',
    'Tarlac': 'Central Luzon (Region III)',
    'Zambales': 'Central Luzon (Region III)',

    # Region IV-A (CALABARZON)
    'Batangas': 'Calabarzon (Region IV-A)',
    'Cavite': 'Calabarzon (Region IV-A)',
    'Laguna': 'Calabarzon (Region IV-A)',
    'Quezon': 'Calabarzon (Region IV-A)',
    'Rizal': 'Calabarzon (Region IV-A)',

    # MIMAROPA Region
    'Marinduque': 'MIMAROPA Region',
    'Occidental Mindoro': 'MIMAROPA Region',
    'Oriental Mindoro': 'MIMAROPA Region',
    'Palawan': 'MIMAROPA Region',
    'Romblon': 'MIMAROPA Region',

    # Region V (Bicol Region)
    'Albay': 'Bicol Region (Region V)',
    'Camarines Norte': 'Bicol Region (Region V)',
    'Camarines Sur': 'Bicol Region (Region V)',
    'Catanduanes': 'Bicol Region (Region V)',
    'Masbate': 'Bicol Region (Region V)',
    'Sorsogon': 'Bicol Region (Region V)',

    # Region VI (Western Visayas)
    'Aklan': 'Western Visayas (Region VI)',
    'Antique': 'Western Visayas (Region VI)',
    'Capiz': 'Western Visayas (Region VI)',
    'Guimaras': 'Western Visayas (Region VI)',
    'Iloilo': 'Western Visayas (Region VI)',
    'Negros Occidental': 'Western Visayas (Region VI)',

    # Region VII (Central Visayas)
    'Bohol': 'Central Visayas (Region VII)',
    'Cebu': 'Central Visayas (Region VII)',
    'Negros Oriental': 'Central Visayas (Region VII)',
    'Siquijor': 'Central Visayas (Region VII)',

    # Region VIII (Eastern Visayas)
    'Biliran': 'Eastern Visayas (Region VIII)',
    'Eastern Samar': 'Eastern Visayas (Region VIII)',
    'Leyte': 'Eastern Visayas (Region VIII)',
    'Northern Samar': 'Eastern Visayas (Region VIII)',
    'Samar': 'Eastern Visayas (Region VIII)',
    'Southern Leyte': 'Eastern Visayas (Region VIII)',

    # Region IX (Zamboanga Peninsula)
    'Zamboanga del Norte': 'Zamboanga Peninsula (Region IX)',
    'Zamboanga del Sur': 'Zamboanga Peninsula (Region IX)',
    'Zamboanga Sibugay': 'Zamboanga Peninsula (Region IX)',

    # Region X (Northern Mindanao)
    'Bukidnon': 'Northern Mindanao (Region X)',
    'Camiguin': 'Northern Mindanao (Region X)',
    'Lanao del Norte': 'Northern Mindanao (Region X)',
    'Misamis Occidental': 'Northern Mindanao (Region X)',
    'Misamis Oriental': 'Northern Mindanao (Region X)',

    # Region XI (Davao Region)
    'Davao de Oro': 'Davao Region (Region XI)',
    'Davao del Norte': 'Davao Region (Region XI)',
    'Davao del Sur': 'Davao Region (Region XI)',
    'Davao Oriental': 'Davao Region (Region XI)',
    'Davao Occidental': 'Davao Region (Region XI)',

    # Region XII (SOCCSKSARGEN)
    'Cotabato': 'SOCCSKSARGEN (Region XII)',
    'Sarangani': 'SOCCSKSARGEN (Region XII)',
    'South Cotabato': 'SOCCSKSARGEN (Region XII)',
    'Sultan Kudarat': 'SOCCSKSARGEN (Region XII)',

    # Region XIII (Caraga)
    'Agusan del Norte': 'Caraga (Region XIII)',
    'Agusan del Sur': 'Caraga (Region XIII)',
    'Dinagat Islands': 'Caraga (Region XIII)',
    'Surigao del Norte': 'Caraga (Region XIII)',
    'Surigao del Sur': 'Caraga (Region XIII)',

    # BARMM (Bangsamoro Autonomous Region in Muslim Mindanao)
    'Basilan': 'BARMM',
    'Lanao del Sur': 'BARMM',
    'Maguindanao del Norte': 'BARMM',
    'Maguindanao del Sur': 'BARMM',
    'Sulu': 'BARMM',
    'Tawi-Tawi': 'BARMM',

    # CAR (Cordillera Administrative Region)
    'Abra': 'CAR',
    'Apayao': 'CAR',
    'Benguet': 'CAR',
    'Ifugao': 'CAR',
    'Kalinga': 'CAR',
    'Mountain Province': 'CAR'
}

In [14]:
df1

,date,time,latitude,longitude,depth_km,mag,coordinates,distance_km,ns,angle_deg,ew,Location,loc_latitude,loc_longitude,city,province,region
0,10/20/2025,19:35:00,7.00,127.36,20,5.7,094 km S 75° E,94,S,75,E,Manay (Davao Oriental),7.210933,126.539914,Manay,Davao Oriental,NaN
1,10/19/2025,9:34:00,6.95,127.36,20,5.2,095 km S 72° E,95,S,72,E,Manay (Davao Oriental),7.210933,126.539914,Manay,Davao Oriental,NaN
2,10/18/2025,20:31:00,8.84,126.45,34,5.5,019 km S 63° E,19,S,63,E,Cagwait (Surigao Del Sur),8.918274,126.298255,Cagwait,Surigao Del Sur,NaN
3,10/17/2025,7:03:00,9.69,126.22,28,6.0,012 km S 33° E,12,S,33,E,General Luna (Surigao Del Norte),9.782584,126.156386,General Luna,Surigao Del Norte,NaN
4,10/14/2025,10:00:00,7.47,127.05,18,5.6,063 km N 63° E,63,N,63,E,Manay (Davao Oriental),7.210933,126.539914,Manay,Davao Oriental,NaN
5,10/13/2025,1:06:00,10.96,123.94,5,5.8,010 km S 24° W,10,S,24,W,City Of Bogo (Cebu),11.051555,124.003546,City Of Bogo,Cebu,NaN
6,10/11/2025,22:32:00,8.94,126.55,24,6.0,028 km N 86° E,28,N,86,E,Cagwait (Surigao Del Sur),8.918274,126.298255,Cagwait,Surigao Del Sur,NaN
7,10/11/2025,18:27:00,7.23,126.91,18,5.5,041 km N 87° E,41,N,87,E,Manay (Davao Oriental),7.210933,126.539914,Manay,Davao Oriental,NaN
8,10/11/2025,17:32:00,15.24,120.23,112,5.1,022 km S 77° E,22,S,77,E,Botolan (Zambales),15.288997,120.024736,Botolan,Zambales,NaN
9,10/10/2025,19:12:00,7.07,126.90,37,6.8,043 km S 68° E,43,S,68,E,Manay (Davao Oriental),7.210933,126.539914,Manay,Davao Oriental,NaN


In [17]:
print("Mapping regions...")
normalized_dict = {str(k).strip().lower(): v for k, v in province_to_region.items()}
valid_provinces = df1['province'].notna()
# Safely initialize or cast the column to 'object' first
if 'region' not in df1.columns:
    df1['region'] = pd.Series(dtype=object)
else:
    df1['region'] = df1['region'].astype(object)
# Now your .loc assignment will work flawlessly
df1.loc[valid_provinces, 'region'] = df1.loc[valid_provinces, 'province'].str.strip().str.lower().map(normalized_dict)


missing_mappings = df1[df1['province'].notna() & df1['region'].isna()]['province'].unique()

if len(missing_mappings) > 0:
    print(f"\n Provinces are missing from the dictionary:")
    for prov in missing_mappings:
        print(f" - '{prov}'")
    print("Please add them to the 'province_to_region' dictionary and run this again.")
else:
    print("\n Pprovinces successfully mapped to a Region.")

# 5. Save the final file
dfSaveFinal = 'Phivolcs_aggregated_final-new5only.csv'
df1.to_csv(dfSaveFinal, index=False)
print(f"Dataset successfully saved as: {dfSaveFinal}\n")

# 6. Display a sample to verify
df1[['city', 'province', 'region']].head(15)

Mapping regions...

 Pprovinces successfully mapped to a Region.
Dataset successfully saved as: Phivolcs_aggregated_final-new5only.csv



,city,province,region
0,Manay,Davao Oriental,Davao Region (Region XI)
1,Manay,Davao Oriental,Davao Region (Region XI)
2,Cagwait,Surigao Del Sur,Caraga (Region XIII)
3,General Luna,Surigao Del Norte,Caraga (Region XIII)
4,Manay,Davao Oriental,Davao Region (Region XI)
5,City Of Bogo,Cebu,Central Visayas (Region VII)
6,Cagwait,Surigao Del Sur,Caraga (Region XIII)
7,Manay,Davao Oriental,Davao Region (Region XI)
8,Botolan,Zambales,Central Luzon (Region III)
9,Manay,Davao Oriental,Davao Region (Region XI)


In [45]:
#Coordinates (bin) + azimuth
def parse_epicenter_coords(coord_string):
    if pd.isna(coord_string):
        return pd.Series([None, None, None, None, None])
    coord_string = str(coord_string).strip().upper()
    match = re.search(r'(\d+(?:\.\d+)?)\s*KM\s+([NS])\s+(\d+(?:\.\d+)?)[^\w]*([EW])', coord_string)
    
    if not match:
        return pd.Series([None, None, None, None, None])
    

    distance = float(match.group(1))
    ns = match.group(2)
    angle = float(match.group(3))
    ew = match.group(4)
    
    # Calculate Azimuth based on the quadrant
    if ns == 'N' and ew == 'E':
        azimuth = angle               # Northeast
    elif ns == 'S' and ew == 'E':
        azimuth = 180 - angle         # Southeast
    elif ns == 'S' and ew == 'W':
        azimuth = 180 + angle         # Southwest
    elif ns == 'N' and ew == 'W':
        azimuth = 360 - angle         # Northwest
    else:
        azimuth = None
        
    return pd.Series([distance, ns, angle, ew, azimuth])



df1[['Distance (km)', 'NS', 'Angle (°)', 'EW', 'Azimuth (°)']] = df1['Coordinates'].apply(parse_epicenter_coords)


print(df1[['Coordinates', 'Distance (km)', 'NS', 'Angle (°)', 'EW', 'Azimuth (°)']].head(10))
dfSaveFinal = 'Phivolcs_aggregated_final.csv'
df1.to_csv(dfSaveFinal, index=False)
print(f"\nDataset successfully updated and saved as: {dfSaveFinal}")

     Coordinates  Distance (km) NS  Angle (°) EW  Azimuth (°)
0  028km N 86° E           28.0  N       86.0  E         86.0
1  041km N 87° E           41.0  N       87.0  E         87.0
2  022km S 77° E           22.0  S       77.0  E        103.0
3  043km S 68° E           43.0  S       68.0  E        112.0
4  065km S 76° E           65.0  S       76.0  E        104.0
5  083km S 78° E           83.0  S       78.0  E        102.0
6  054km S 68° E           54.0  S       68.0  E        112.0
7  048km N 89° E           48.0  N       89.0  E         89.0
8  052km N 71° W           52.0  N       71.0  W        289.0
9  019km N 68° E           19.0  N       68.0  E         68.0

Dataset successfully updated and saved as: Phivolcs_aggregated_final.csv


#FinalFixes

In [19]:
import pandas as pd

df2 = pd.read_csv("Phivolcs_aggregated_final_list.csv")

df2['date'] = pd.to_datetime(df2['date'], format='%d-%b-%y')
df2['date'] = df2['date'].dt.strftime('%#m/%#d/%Y')
df2[['date']].head(15)

,date
0,10/11/2025
1,10/11/2025
2,10/11/2025
3,10/10/2025
4,10/10/2025
5,10/10/2025
6,10/10/2025
7,10/10/2025
8,10/7/2025
9,10/3/2025


In [ ]:
df2.to_csv("phivolcs_aggregated_final_list_fixed.csv", index=False)
print("Saved successfully!")

Saved successfully!


In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter    
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
import re

df3 = pd.read_csv("Phivolcs_aggregated_final-new5only.csv")
# Sort chronologically by telling pandas to interpret 'date' as time
# ascending=False puts Newest (2026) at the top, and Oldest (2025) at the bottom!
df3 = df3.sort_values(
    by='date', 
    key=pd.to_datetime, 
    ascending=False
).reset_index(drop=True)
df3



,date,time,latitude,longitude,depth_km,mag,coordinates,Location,loc_latitude,loc_longitude,city,province,region,distance_km,ns,angle_deg,ew
0,2/15/2026,1:22:00,5.31,127.44,18,5.3,209 km S 71° E,Jose Abad Santos (Davao Occidental),5.912465,125.644593,Jose Abad Santos,Davao Occidental,Davao Region (Region XI),209,S,71,E
1,2/13/2026,22:56:00,3.95,128.62,53,5.6,387 km S 64° E,Sarangani Island (Municipality Of Sarangani) (...,5.442653,125.475756,Sarangani,Davao Occidental,Davao Region (Region XI),387,S,64,E
2,2/13/2026,23:55:00,4.22,127.84,62,5.1,298 km S 62° E,Sarangani Island (Municipality Of Sarangani) (...,5.442653,125.475756,Sarangani,Davao Occidental,Davao Region (Region XI),298,S,62,E
3,2/12/2026,6:27:00,20.60,119.99,32,5.3,194 km S 83° W,Itbayat (Batanes),20.787814,121.841474,Itbayat,Batanes,Cagayan Valley (Region II),194,S,83,W
4,1/28/2026,16:34:00,6.49,123.67,16,5.2,043 km S 80° W,Kalamansig (Sultan Kudarat),6.554584,124.050599,Kalamansig,Sultan Kudarat,SOCCSKSARGEN (Region XII),43,S,80,W
5,1/28/2026,14:46:00,6.52,123.69,4,5.9,040 km S 84° W,Kalamansig (Sultan Kudarat),6.554584,124.050599,Kalamansig,Sultan Kudarat,SOCCSKSARGEN (Region XII),40,S,84,W
6,1/28/2026,9:41:00,6.47,123.73,2,5.2,037 km S 76° W,Kalamansig (Sultan Kudarat),6.554584,124.050599,Kalamansig,Sultan Kudarat,SOCCSKSARGEN (Region XII),37,S,76,W
7,1/28/2026,3:54:00,6.43,123.67,6,5.1,045 km S 72° W,Kalamansig (Sultan Kudarat),6.554584,124.050599,Kalamansig,Sultan Kudarat,SOCCSKSARGEN (Region XII),45,S,72,W
8,1/28/2026,1:05:00,6.42,123.68,14,5.4,043 km S 70° W,Kalamansig (Sultan Kudarat),6.554584,124.050599,Kalamansig,Sultan Kudarat,SOCCSKSARGEN (Region XII),43,S,70,W
9,1/28/2026,16:38:00,6.49,123.67,13,5.2,042 km S 81° W,Kalamansig (Sultan Kudarat),6.554584,124.050599,Kalamansig,Sultan Kudarat,SOCCSKSARGEN (Region XII),42,S,81,W


In [2]:
df3.to_csv("Phivolcs_aggregated_final-new5only-fixed.csv", index=False)
print("Saved perfectly!")

Saved perfectly!


#combinatronics

In [3]:

df1 = pd.read_csv("phivolcs_aggregated_final_list_fixed.csv")
df2 = pd.read_csv("Phivolcs_aggregated_final-new5only-fixed.csv")
# 2. Stack them together into one large DataFrame
df_combined = pd.concat([df1, df2], ignore_index=True)
# 3. Sort the newly combined dates chronologically (Newest to Oldest)
df_combined = df_combined.sort_values(
    by='date', 
    key=pd.to_datetime, 
    ascending=False
).reset_index(drop=True)
# 4. Save to your final master file
save_name = "Phivolcs_Master_Aggregated_Fixed.csv"
df_combined.to_csv(save_name, index=False)
print(f"Successfully combined {len(df_combined)} total rows into {save_name}!")
df_combined.head()

Successfully combined 771 total rows into Phivolcs_Master_Aggregated_Fixed.csv!


,date,time,latitude,longitude,depth_km,mag,coordinates,Location,loc_latitude,loc_longitude,city,province,region,distance_km,ns,angle_deg,ew
0,2/15/2026,1:22:00,5.31,127.44,18,5.3,209 km S 71° E,Jose Abad Santos (Davao Occidental),5.912465,125.644593,Jose Abad Santos,Davao Occidental,Davao Region (Region XI),209.0,S,71.0,E
1,2/13/2026,22:56:00,3.95,128.62,53,5.6,387 km S 64° E,Sarangani Island (Municipality Of Sarangani) (...,5.442653,125.475756,Sarangani,Davao Occidental,Davao Region (Region XI),387.0,S,64.0,E
2,2/13/2026,23:55:00,4.22,127.84,62,5.1,298 km S 62° E,Sarangani Island (Municipality Of Sarangani) (...,5.442653,125.475756,Sarangani,Davao Occidental,Davao Region (Region XI),298.0,S,62.0,E
3,2/12/2026,6:27:00,20.60,119.99,32,5.3,194 km S 83° W,Itbayat (Batanes),20.787814,121.841474,Itbayat,Batanes,Cagayan Valley (Region II),194.0,S,83.0,W
4,1/28/2026,16:34:00,6.49,123.67,16,5.2,043 km S 80° W,Kalamansig (Sultan Kudarat),6.554584,124.050599,Kalamansig,Sultan Kudarat,SOCCSKSARGEN (Region XII),43.0,S,80.0,W


In [4]:

df_master = pd.read_csv("Phivolcs_Master_Aggregated_Fixed.csv")

event_ids = ['PHV-' + str(i).zfill(4) for i in range(1, len(df_master) + 1)]

df_master.insert(0, 'event_id', event_ids)

save_name = "Phivolcs_Master_Aggregated_Fixed_With_IDs.csv"
df_master.to_csv(save_name, index=False)
print(f"Added Event IDs and saved to {save_name}!")
# Verify the first few rows to see the new ID column
df_master.head()

Added Event IDs and saved to Phivolcs_Master_Aggregated_Fixed_With_IDs.csv!


,event_id,date,time,latitude,longitude,depth_km,mag,coordinates,Location,loc_latitude,loc_longitude,city,province,region,distance_km,ns,angle_deg,ew
0,PHV-0001,2/15/2026,1:22:00,5.31,127.44,18,5.3,209 km S 71° E,Jose Abad Santos (Davao Occidental),5.912465,125.644593,Jose Abad Santos,Davao Occidental,Davao Region (Region XI),209.0,S,71.0,E
1,PHV-0002,2/13/2026,22:56:00,3.95,128.62,53,5.6,387 km S 64° E,Sarangani Island (Municipality Of Sarangani) (...,5.442653,125.475756,Sarangani,Davao Occidental,Davao Region (Region XI),387.0,S,64.0,E
2,PHV-0003,2/13/2026,23:55:00,4.22,127.84,62,5.1,298 km S 62° E,Sarangani Island (Municipality Of Sarangani) (...,5.442653,125.475756,Sarangani,Davao Occidental,Davao Region (Region XI),298.0,S,62.0,E
3,PHV-0004,2/12/2026,6:27:00,20.60,119.99,32,5.3,194 km S 83° W,Itbayat (Batanes),20.787814,121.841474,Itbayat,Batanes,Cagayan Valley (Region II),194.0,S,83.0,W
4,PHV-0005,1/28/2026,16:34:00,6.49,123.67,16,5.2,043 km S 80° W,Kalamansig (Sultan Kudarat),6.554584,124.050599,Kalamansig,Sultan Kudarat,SOCCSKSARGEN (Region XII),43.0,S,80.0,W


In [5]:
# Load the file
df_master = pd.read_csv("Phivolcs_Master_Aggregated_Fixed_With_IDs.csv")

missing_mask = df_master.isna().any(axis=1)

missing_event_ids = df_master.loc[missing_mask, 'event_id'].tolist()

print(f"Total events with missing data: {len(missing_event_ids)}")
if len(missing_event_ids) > 0:
    print("\nHere are their Event IDs:")
    print(missing_event_ids)
else:
    print("\n No missings")

Total events with missing data: 46

Here are their Event IDs:
['PHV-0319', 'PHV-0360', 'PHV-0375', 'PHV-0455', 'PHV-0478', 'PHV-0526', 'PHV-0547', 'PHV-0565', 'PHV-0583', 'PHV-0594', 'PHV-0613', 'PHV-0620', 'PHV-0621', 'PHV-0662', 'PHV-0665', 'PHV-0669', 'PHV-0683', 'PHV-0697', 'PHV-0744', 'PHV-0745', 'PHV-0746', 'PHV-0747', 'PHV-0748', 'PHV-0749', 'PHV-0750', 'PHV-0751', 'PHV-0752', 'PHV-0753', 'PHV-0754', 'PHV-0755', 'PHV-0756', 'PHV-0757', 'PHV-0758', 'PHV-0759', 'PHV-0760', 'PHV-0761', 'PHV-0762', 'PHV-0763', 'PHV-0764', 'PHV-0765', 'PHV-0766', 'PHV-0767', 'PHV-0768', 'PHV-0769', 'PHV-0770', 'PHV-0771']
